# Image Classification with MobileNetV3 Small (timm)

This notebook uses the `timm/mobilenetv3_small_100.lamb_in1k` model to classify images in the `images` folder.

In [ ]:
import os
import torch
import timm
import json
import requests
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load Model and Configuration

In [ ]:
model_name = 'mobilenetv3_small_100.lamb_in1k'
model = timm.create_model(model_name, pretrained=True)
model.eval()

# Get model specific configuration
config = model.default_cfg
print(f"Model: {model_name} loaded successfully.")
print(f"Input shape: {config['input_size']}")

## 2. Setup ImageNet Classes

In [ ]:
# Download ImageNet labels
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(labels_url)
labels = [line.strip() for line in response.text.split('\n')]

print(f"Loaded {len(labels)} class labels.")

## 3. Preprocessing and Inference Function

In [ ]:
transform = transforms.Compose([
    transforms.Resize(config['input_size'][1:]), 
    transforms.CenterCrop(config['input_size'][1:]),
    transforms.ToTensor(),
    transforms.Normalize(config['mean'], config['std']),
])

def classify_image(image_path, topk=5):
    """Extracts the top-K predictions for a given image."""
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0)
    
    with torch.no_grad():
        output = model(input_tensor)
        
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top_prob, top_catid = torch.topk(probabilities, topk)
    
    return [(labels[top_catid[i].item()], top_prob[i].item()) for i in range(topk)]

## 4. Classify Images in 'images' folder

In [ ]:
images_dir = 'images'
image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if not image_files:
    print(f"No images found in {images_dir}.")
else:
    n_images = len(image_files)
    # Reduzindo o figsize total e forçando constrained_layout
    fig, axes = plt.subplots(n_images, 2, figsize=(14, 4.8 * n_images), 
                             constrained_layout=True)
    
    if n_images == 1:
        axes = [axes]
        
    for i, filename in enumerate(image_files):
        image_path = os.path.join(images_dir, filename)
        top5 = classify_image(image_path)
        labels_list, probs_list = zip(*top5)
        
        # 1. Image Display - Sem margem superior no eixo
        ax_img = axes[i][0]
        img = Image.open(image_path)
        ax_img.imshow(img)
        ax_img.set_title(f"FILE: {filename}", fontsize=12, fontweight='bold')
        ax_img.axis('off')
        
        # 2. Probability Chart
        ax_bar = axes[i][1]
        y_pos = range(len(labels_list))
        colors = plt.cm.Blues(torch.linspace(0.8, 0.4, 5))
        bars = ax_bar.barh(y_pos, [p * 100 for p in probs_list], color=colors, edgecolor='black', alpha=0.8)
        
        ax_bar.set_yticks(y_pos)
        ax_bar.set_yticklabels(labels_list, fontsize=10, fontweight='medium')
        ax_bar.invert_yaxis()
        ax_bar.set_xlabel('Confidence (%)', fontsize=10)
        ax_bar.set_xlim(0, 115) 
        ax_bar.spines['top'].set_visible(False)
        ax_bar.spines['right'].set_visible(False)
        ax_bar.grid(axis='x', linestyle=':', alpha=0.4)
        
        for bar in bars:
            width = bar.get_width()
            ax_bar.text(width + 1.2, bar.get_y() + bar.get_height()/2, 
                        f'{width:.2f}%', va='center', fontsize=9, fontweight='bold')

    plt.show()